# Notebook 00 — Baseline Evaluation
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**What this notebook does:**
1. Defines 30 reasoning prompts:
   - 5 general logic (G1–G5)
   - 5 math word problems (M1–M5)
   - 10 difficult reasoning questions (D1–D10)
   - 10 Python coding questions (C1–C10)
2. Gold answers pre-filled from ChatGPT
3. Runs both `Qwen3-0.6B` and `Qwen3-1.7B` base models **locally** (GPU if available, else CPU)
4. Scores all responses using **Kimi-K2** as LLM-as-Judge via the HF OpenAI-compatible router
5. Saves results to `baseline_results.csv`, `prompts.json`, `gold_answers.json`

The expanded prompt set helps identify which category each dataset trains best on, guiding dataset selection for SFT.

**Run this on:** Any machine — uses local inference for base models, HF router API for judge only

In [8]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q huggingface_hub pandas transformers torch accelerate python-dotenv openai

In [9]:
import os, json, time, re, pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ── CONFIG — fill in your HF token ───────────────────────────────────────────
HF_TOKEN = os.environ["HF_TOKEN"]

BASE_MODELS = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B",
]

# Judge must differ from gold-answer generator (assignment rule)
# Gold answers: generated via ChatGPT (pasted below)
# Judge: Kimi via HF OpenAI-compatible router (different from ChatGPT gold generator)
# Mistral-7B-Instruct-v0.3 currently fails on HF provider routing for this account.
JUDGE_MODEL = "moonshotai/Kimi-K2-Instruct:novita"
JUDGE_CLIENT = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)

# Alternative: if 3090 Ti has Ollama set up with Jackrong's 27B model:
# Use ollama python client instead — swap judge calls at the bottom

## Step 1 — Define 30 Reasoning Prompts
- **G1–G5**: General logic puzzles
- **M1–M5**: Math word problems
- **D1–D10**: Difficult reasoning (harder logic, probability, combinatorics, number theory)
- **C1–C10**: Python coding problems

These same 30 prompts are used throughout ALL notebooks for evaluation.
The coding + difficult categories help reveal whether Dataset A (reasoning traces) or Dataset B (math solutions) generalises better across problem types.

In [ ]:
PROMPTS = [
    # ── General Logic (5) ────────────────────────────────────────────────────
    {
        "id": "G1",
        "type": "general",
        "prompt": (
            "Alice, Bob, and Carol each own exactly one pet: a cat, a dog, or a fish. "
            "Alice does not own the cat. Bob does not own the dog. "
            "Carol does not own the fish. Who owns which pet? Reason step by step."
        )
    },
    {
        "id": "G2",
        "type": "general",
        "prompt": (
            "A bat and a ball together cost $1.10. The bat costs $1.00 more than the ball. "
            "How much does the ball cost? Show your reasoning carefully."
        )
    },
    {
        "id": "G3",
        "type": "general",
        "prompt": (
            "In a room of 30 people, is it more likely than not that at least two share a birthday? "
            "Explain your reasoning step by step without necessarily computing the exact probability."
        )
    },
    {
        "id": "G4",
        "type": "general",
        "prompt": (
            "If all Bloops are Razzies, and all Razzies are Lazzies, are all Bloops definitely Lazzies? "
            "What logical principle applies here? Explain step by step."
        )
    },
    {
        "id": "G5",
        "type": "general",
        "prompt": (
            "A farmer has 17 sheep. All but 9 die. How many sheep are left? "
            "Explain your reasoning before giving the answer."
        )
    },
    # ── Math Word Problems (5) ────────────────────────────────────────────────
    {
        "id": "M1",
        "type": "math",
        "prompt": (
            "A train travels at 60 km/h for 2 hours, then at 90 km/h for 1.5 hours. "
            "What is the total distance travelled? Show all working."
        )
    },
    {
        "id": "M2",
        "type": "math",
        "prompt": (
            "Find the sum of all integers from 1 to 100. "
            "Show the formula you use and explain why it works."
        )
    },
    {
        "id": "M3",
        "type": "math",
        "prompt": (
            "A store sells an item at a 20% discount. After the discount the price is $80. "
            "What was the original price? Show step-by-step working."
        )
    },
    {
        "id": "M4",
        "type": "math",
        "prompt": (
            "Two pipes can fill a tank in 6 hours and 4 hours respectively. "
            "If both pipes are opened simultaneously, how long will it take to fill the tank? "
            "Show all reasoning."
        )
    },
    {
        "id": "M5",
        "type": "math",
        "prompt": (
            "A sequence starts: 2, 6, 18, 54, ... "
            "What is the 8th term? Identify the pattern and show your working."
        )
    },
    # ── Difficult Reasoning (10) ──────────────────────────────────────────────
    {
        "id": "D1",
        "type": "difficult",
        "prompt": (
            "On an island, knights always tell the truth and knaves always lie. "
            "You meet two inhabitants A and B. A says: 'At least one of us is a knave.' "
            "Is A a knight or a knave? What about B? Reason step by step."
        )
    },
    {
        "id": "D2",
        "type": "difficult",
        "prompt": (
            "You are on a game show. There are 3 doors: behind one is a car, behind the other two are goats. "
            "You pick door 1. The host, who knows what is behind each door, opens door 3 to reveal a goat. "
            "He offers you the chance to switch to door 2. Should you switch? "
            "What is the probability of winning if you switch versus if you stay? Reason carefully."
        )
    },
    {
        "id": "D3",
        "type": "difficult",
        "prompt": (
            "How many times do the hour hand and minute hand of a clock overlap "
            "(point in exactly the same direction) in a 24-hour period? "
            "Derive the answer mathematically rather than counting."
        )
    },
    {
        "id": "D4",
        "type": "difficult",
        "prompt": (
            "A rubber ball is dropped from a height of 100 metres. "
            "Each time it bounces, it rises to two-thirds of the height from which it just fell. "
            "What is the total distance (up and down combined) the ball travels before coming to rest? "
            "Show all working and simplify your answer."
        )
    },
    {
        "id": "D5",
        "type": "difficult",
        "prompt": (
            "A disease affects 1% of the population. A diagnostic test is 99% accurate: "
            "if you have the disease the test is positive 99% of the time; "
            "if you do not have the disease the test is negative 99% of the time. "
            "You test positive. What is the probability you actually have the disease? "
            "Show all working using Bayes' theorem."
        )
    },
    {
        "id": "D6",
        "type": "difficult",
        "prompt": (
            "You flip a fair coin repeatedly until you get heads. "
            "You win $2^n where n is the number of flips required. "
            "What is the expected value of your winnings? "
            "Analyse the result and explain why it is surprising (this is the St. Petersburg Paradox)."
        )
    },
    {
        "id": "D7",
        "type": "difficult",
        "prompt": (
            "In how many ways can 8 people be seated around a circular table? "
            "Now suppose two specific people, Alice and Bob, must NOT sit next to each other. "
            "How many valid arrangements remain? Show all working."
        )
    },
    {
        "id": "D8",
        "type": "difficult",
        "prompt": (
            "A snail is at the bottom of a 10-metre well. "
            "Each day it climbs 3 metres, but each night it slides back 2 metres. "
            "How many days does it take to reach the top? "
            "Now generalise: for a well of height h metres, daily climb c metres, nightly slide s metres "
            "(with c > s and c > h impossible after partial days), derive a formula for the number of days."
        )
    },
    {
        "id": "D9",
        "type": "difficult",
        "prompt": (
            "Alice and Bob take turns rolling a fair six-sided die. Alice goes first. "
            "The first person to roll a 6 wins. What is the probability that Alice wins? "
            "Express your answer as a simplified fraction and show all working."
        )
    },
    {
        "id": "D10",
        "type": "difficult",
        "prompt": (
            "Prove that the square root of 2 is irrational using proof by contradiction. "
            "Then state whether the square root of any non-perfect-square positive integer is always "
            "irrational, and justify your answer."
        )
    },
    # ── Python Coding Problems (10) ───────────────────────────────────────────
    {
        "id": "C1",
        "type": "coding",
        "prompt": (
            "Write a Python function is_prime(n) that returns True if n is prime and False otherwise. "
            "It must handle all non-negative integers correctly. "
            "Explain your approach and state its time complexity."
        )
    },
    {
        "id": "C2",
        "type": "coding",
        "prompt": (
            "Write a Python function fib(n) that returns the nth Fibonacci number "
            "(fib(0)=0, fib(1)=1). Implement it using memoization. "
            "Explain how memoization improves the time complexity compared to naive recursion, "
            "and state the complexity of both approaches."
        )
    },
    {
        "id": "C3",
        "type": "coding",
        "prompt": (
            "Write a Python function two_sum(nums, target) that takes a list of integers and a target, "
            "and returns the indices [i, j] of the two numbers that add up to the target. "
            "Assume exactly one solution exists and you cannot use the same element twice. "
            "Achieve O(n) time complexity and explain how."
        )
    },
    {
        "id": "C4",
        "type": "coding",
        "prompt": (
            "Write a Python function binary_search(arr, target) that searches for target in a sorted list. "
            "Return the index if found, or -1 if not found. "
            "Do not use any built-in search functions. "
            "Explain why binary search achieves O(log n) time complexity."
        )
    },
    {
        "id": "C5",
        "type": "coding",
        "prompt": (
            "Write a Python function is_balanced(s) that takes a string containing only the characters "
            "( ) { } [ ] and returns True if all brackets are correctly nested and matched, False otherwise. "
            "Explain your approach and its time complexity."
        )
    },
    {
        "id": "C6",
        "type": "coding",
        "prompt": (
            "Define a Python class ListNode for a singly linked list node (with val and next attributes). "
            "Then write a function reverse_linked_list(head) that reverses the list in-place and returns "
            "the new head. Walk through an example with the list 1 → 2 → 3 → 4 to verify your solution."
        )
    },
    {
        "id": "C7",
        "type": "coding",
        "prompt": (
            "Write a Python function permutations(lst) that returns a list of all permutations of the "
            "input list of unique integers. Do not use itertools. "
            "Explain the recursive approach and state its time complexity."
        )
    },
    {
        "id": "C8",
        "type": "coding",
        "prompt": (
            "Write a Python function merge_sorted(a, b) that takes two sorted lists of integers and "
            "returns a new sorted list containing all elements from both. "
            "Do not use sort() or sorted(). Achieve O(m+n) time complexity and explain the approach."
        )
    },
    {
        "id": "C9",
        "type": "coding",
        "prompt": (
            "Write a Python function shortest_path(grid, start, end) where grid is a 2D list of 0s "
            "(passable) and 1s (walls), and start/end are (row, col) tuples. "
            "Return the number of steps in the shortest path from start to end, or -1 if no path exists. "
            "Use BFS and explain why BFS guarantees the shortest path."
        )
    },
    {
        "id": "C10",
        "type": "coding",
        "prompt": (
            "Write a Python function calculate(expression) that evaluates a string arithmetic expression "
            "containing non-negative integers and the operators +, -, *, // (integer division). "
            "The expression may contain spaces. Respect standard operator precedence (* and // before + and -). "
            "For example: calculate('3 + 5 * 2') should return 13. "
            "Explain your approach."
        )
    },
]

type_counts = {}
for p in PROMPTS:
    type_counts[p["type"]] = type_counts.get(p["type"], 0) + 1
print(f"Defined {len(PROMPTS)} prompts: {type_counts}")

## Step 2 — Gold Answers (generated via ChatGPT)

Reference answers for all 30 prompts. Keys match prompt IDs.
- G1–G5: general logic gold answers
- M1–M5: math gold answers
- D1–D10: difficult reasoning gold answers
- C1–C10: coding gold answers

In [ ]:
gold_answers = {
    # ── General Logic ──────────────────────────────────────────────────────────
    "G1": """We'll solve it step by step.

We have:
- Pets: cat, dog, fish
- People: Alice, Bob, Carol
- Each person owns exactly one different pet.

Clues:
1. Alice does not own the cat.
2. Bob does not own the dog.
3. Carol does not own the fish.

Step 1: Suppose Alice owns the dog.
Then the remaining pets are cat and fish for Bob and Carol.
- Carol cannot own the fish → Carol must own the cat, and Bob must own the fish.
This works: Alice → dog, Bob → fish, Carol → cat.

Step 2: Check the other possibility.
Suppose Alice owns the fish.
Then Bob and Carol have cat and dog.
- Bob cannot own the dog → Bob must own the cat, and Carol owns the dog.
This also satisfies all clues: Alice → fish, Bob → cat, Carol → dog.

So the information does not yield a unique answer. The valid solutions are:
1. Alice → dog, Bob → fish, Carol → cat
2. Alice → fish, Bob → cat, Carol → dog""",

    "G2": """Let the ball cost x dollars.
Then the bat costs x + 1.00 (because the bat costs $1 more than the ball).

Add them together:
x + (x + 1.00) = 1.10
2x + 1.00 = 1.10
2x = 0.10
x = 0.05

So the ball costs $0.05.
And the bat costs $1.05, which together make $1.10.""",

    "G3": """Yes — in a room of 30 people, it is more likely than not that at least two share a birthday. This is the famous "birthday paradox."

Step 1: Think about the opposite event.
Ask "What is the chance that everyone has different birthdays?" If that probability drops below 50%, the probability of at least one match must be above 50%.

Step 2: Build the probability person by person.
Assume 365 equally likely birthdays and ignore leap years.
- Person 1: any birthday: 365/365 = 1
- Person 2: must avoid person 1: 364/365
- Person 3: must avoid the first two: 363/365
- ...continuing to person 30: 336/365
So P(all different) = (365 × 364 × ... × 336) / 365^30 ≈ 0.294

Step 3: Probability of at least one shared birthday = 1 − 0.294 ≈ 70.6%

Step 4: Intuition.
With 30 people there are (30 × 29)/2 = 435 different pairs — a huge number of opportunities for a match. By 23 people, the probability is already above 50%.

Answer: Yes, with 30 people there is roughly a 70.6% chance that two people share a birthday.""",

    "G4": """Yes — all Bloops are definitely Lazzies.

Step 1: Write the statements clearly.
1. All Bloops are Razzies. (Bloops ⊆ Razzies)
2. All Razzies are Lazzies. (Razzies ⊆ Lazzies)

Step 2: Follow the chain.
Take any Bloop.
- Since all Bloops are Razzies, that Bloop must be a Razzie.
- Since all Razzies are Lazzies, that Razzie must also be a Lazzie.
Therefore: every Bloop is a Lazzie.

Step 3: The logical principle.
This uses the transitive property of class inclusion (syllogism).
General form: If A ⊆ B and B ⊆ C, then A ⊆ C.

Step 4: Simple analogy.
All cats are mammals. All mammals are animals. Therefore all cats are animals. Same structure.""",

    "G5": """Let's read the sentence carefully: "A farmer has 17 sheep. All but 9 die."

The key phrase is "All but 9." This means every sheep dies except 9 sheep. So 9 sheep survive.

Another way to think about it:
- Start with 17 sheep.
- "All but 9 die" = all the others die, leaving exactly 9.

The answer is 9.""",

    # ── Math Word Problems ─────────────────────────────────────────────────────
    "M1": """Use the basic formula: Distance = Speed × Time

Step 1: First part of the journey.
Speed = 60 km/h, Time = 2 hours
Distance = 60 × 2 = 120 km

Step 2: Second part of the journey.
Speed = 90 km/h, Time = 1.5 hours
Distance = 90 × 1.5 = 135 km

Step 3: Add the distances.
120 + 135 = 255 km

The total distance travelled is 255 km.""",

    "M2": """We want to find: 1 + 2 + 3 + ... + 100

Use the formula for the sum of the first n integers: n(n+1)/2
Here, n = 100.

So: 100 × 101 / 2 = 50 × 101 = 5050

The sum is 5050.

Why the formula works:
Write the numbers forward and backward:
1 + 2 + ... + 99 + 100
100 + 99 + ... + 2 + 1

Add vertically: each pair equals 101. There are 100 such pairs: 100 × 101 = 10100.
Divide by 2 (counted twice): 10100 / 2 = 5050.""",

    "M3": """Let the original price be x.
A 20% discount means the customer pays 80% of the original price.
0.8x = 80
x = 80 / 0.8
x = 100

The original price was $100.

Check: 20% of $100 = $20. After discount: 100 − 20 = $80. Correct.""",

    "M4": """Step 1: Find each pipe's filling rate.
Pipe A fills the tank in 6 hours → 1/6 of the tank per hour.
Pipe B fills the tank in 4 hours → 1/4 of the tank per hour.

Step 2: Combined rate.
1/6 + 1/4 = 2/12 + 3/12 = 5/12 per hour.

Step 3: Total time = 1 ÷ (5/12) = 12/5 = 2.4 hours = 2 hours 24 minutes.""",

    "M5": """The sequence is: 2, 6, 18, 54, ...

Step 1: Identify the pattern.
2 × 3 = 6, 6 × 3 = 18, 18 × 3 = 54. Each term is multiplied by 3.
This is a geometric sequence with first term a = 2 and common ratio r = 3.

Step 2: Use the formula: a_n = a × r^(n−1)
For the 8th term: a_8 = 2 × 3^7 = 2 × 2187 = 4374

The 8th term is 4374.""",

    # ── Difficult Reasoning ────────────────────────────────────────────────────
    "D1": """Step 1: Assume A is a knave.
If A is a knave, A always lies. A's statement "At least one of us is a knave" would be false.
For it to be false, neither A nor B is a knave — both are knights.
But that contradicts our assumption that A is a knave. Contradiction.

Step 2: Therefore A must be a knight.
A tells the truth, so "At least one of us is a knave" is true.
Since A is a knight, the knave must be B.

Answer: A is a knight, B is a knave.""",

    "D2": """You should switch. Here is why:

Step 1: Initial probabilities.
When you pick door 1, the car is behind door 1 with probability 1/3, and behind doors 2 or 3 with probability 2/3.

Step 2: The host's action is not random.
The host knows where the car is and always opens a goat door. This is key — his action transfers information.

Step 3: After the host opens door 3 (a goat):
- Probability the car is behind door 1 (your original pick): still 1/3.
- Probability the car is behind door 2: 2/3 (the entire 2/3 probability that was on doors 2 and 3 collapses onto door 2 since door 3 is eliminated).

Step 4: Conclusion.
Stay: win with probability 1/3.
Switch: win with probability 2/3.

You should always switch. Switching wins twice as often as staying.""",

    "D3": """Step 1: Set up the mathematics.
Angular speeds:
- Minute hand: 360°/60 min = 6°/min
- Hour hand: 360°/720 min = 0.5°/min
- Relative speed of minute hand gaining on hour hand: 6 − 0.5 = 5.5°/min

Step 2: Time between overlaps.
The hands overlap when the minute hand has gained exactly 360° on the hour hand.
Time between overlaps = 360 / 5.5 = 720/11 minutes ≈ 65.45 minutes.

Step 3: Overlaps in 12 hours.
Number of overlaps = (12 × 60) / (720/11) = 720 × 11 / 720 = 11.

Step 4: In 24 hours.
11 × 2 = 22 overlaps.

The hour and minute hands overlap exactly 22 times in a 24-hour period.""",

    "D4": """Step 1: The first drop.
The ball falls 100 m.

Step 2: Each subsequent bounce.
After bounce k the ball rises to 100 × (2/3)^k metres then falls the same distance.
Each bounce contributes: 2 × 100 × (2/3)^k

Step 3: Sum the infinite geometric series.
Total = 100 + Σ[k=1 to ∞] 2 × 100 × (2/3)^k
     = 100 + 200 × (2/3) / (1 − 2/3)
     = 100 + 200 × (2/3) / (1/3)
     = 100 + 200 × 2
     = 100 + 400
     = 500 metres

The ball travels a total distance of 500 metres.""",

    "D5": """Using Bayes' Theorem: P(Disease | Positive) = P(Positive | Disease) × P(Disease) / P(Positive)

Step 1: Define the values.
- P(Disease) = 0.01
- P(No disease) = 0.99
- P(Positive | Disease) = 0.99  (sensitivity)
- P(Positive | No disease) = 0.01  (false positive rate)

Step 2: Calculate P(Positive) by total probability.
P(Positive) = 0.99 × 0.01 + 0.01 × 0.99 = 0.0099 + 0.0099 = 0.0198

Step 3: Apply Bayes' theorem.
P(Disease | Positive) = (0.99 × 0.01) / 0.0198 = 0.0099 / 0.0198 = 0.5

Despite the 99% accurate test, a positive result means only a 50% chance of actually having the disease.
Why: the disease is rare (1%), so false positives are exactly as numerous as true positives among all positive tests.""",

    "D6": """Step 1: Calculate the expected value.
Let n be the number of flips until the first heads. P(n = k) = (1/2)^k.
Winnings = 2^k.

E = Σ[k=1 to ∞] 2^k × (1/2)^k = Σ[k=1 to ∞] 1 = ∞

The expected value is infinite.

Step 2: Why this is surprising (the St. Petersburg Paradox).
Despite the infinite expected value, most people would not pay more than $20–$30 to play this game.

Step 3: Explanation.
- Standard expected-value theory says a rational agent should pay any finite amount to play.
- In practice, large payouts (e.g. $2^40) have astronomically small probabilities, so they barely affect real decisions.
- The paradox reveals that raw expected value is a poor decision criterion when outcomes have extreme variance — real decisions require accounting for diminishing marginal utility of wealth.""",

    "D7": """Step 1: Total circular arrangements.
In circular permutations, one person is fixed to remove rotational equivalence.
For 8 people: (8−1)! = 7! = 5040 arrangements.

Step 2: Arrangements where Alice and Bob ARE adjacent.
Treat Alice+Bob as a single unit → 7 units in a circle: (7−1)! = 6! = 720.
Alice and Bob can be in 2 internal orders: 720 × 2 = 1440.

Step 3: Arrangements where Alice and Bob are NOT adjacent.
5040 − 1440 = 3600.

Answers:
- Total arrangements of 8 people around a circular table: 5040
- Arrangements with Alice and Bob not sitting next to each other: 3600""",

    "D8": """Part 1: The 10-metre well.
Net progress per full day-night cycle = 3 − 2 = 1 m.
After 7 complete cycles the snail is at 7 m.
On day 8 it climbs 3 m, reaching 10 m — it escapes during the day and does not slide back.

Answer: 8 days.

Part 2: General formula.
The snail needs to reach height h. On the final day it climbs c metres from some height h_prev ≥ h − c.
Before the final day it completes full cycles of net (c − s) metres each.

Number of full cycles needed before the last day: ceil((h − c) / (c − s))
Total days = ceil((h − c) / (c − s)) + 1

Verification: h=10, c=3, s=2: ceil(7/1) + 1 = 7 + 1 = 8. ✓""",

    "D9": """Let P(A) = probability that Alice wins.

Step 1: Alice wins on her first roll with probability 1/6.
If Alice fails (prob 5/6) and Bob also fails (prob 5/6), we are back to Alice's turn with the same probability P(A).

Step 2: Write the equation.
P(A) = 1/6 + (5/6)(5/6) × P(A)
P(A) = 1/6 + (25/36) × P(A)
P(A)(1 − 25/36) = 1/6
P(A)(11/36) = 1/6
P(A) = (1/6) × (36/11) = 6/11

Alice wins with probability 6/11 ≈ 54.5%.
Bob wins with probability 5/11 ≈ 45.5%.

Sense check: Alice goes first so she has an advantage, and 6/11 > 1/2. ✓""",

    "D10": """Proof that √2 is irrational (by contradiction):

Assume √2 is rational, i.e. √2 = p/q where p, q are integers in lowest terms (no common factors) and q ≠ 0.

Then: 2 = p²/q²  →  p² = 2q²

So p² is even → p must be even (odd² is always odd). Write p = 2k.

Then: (2k)² = 2q²  →  4k² = 2q²  →  q² = 2k²

So q² is even → q must also be even.

But p and q both even means they share the factor 2, contradicting the assumption that p/q is in lowest terms. □

Generalisation:
Yes, √n is irrational for every positive integer n that is not a perfect square.
The same argument applies: if √n = p/q in lowest terms then p² = nq². By analysing prime factorisations, every prime factor of n must appear an even number of times in n for n to be a perfect square. If n has any prime factor with an odd exponent, the same contradiction arises, forcing common factors in p and q. Hence √n is irrational for all non-perfect-square n.""",

    # ── Python Coding ──────────────────────────────────────────────────────────
    "C1": """def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    i = 3
    while i * i <= n:
        if n % i == 0:
            return False
        i += 2
    return True

Approach:
- n < 2: not prime by definition.
- n == 2: the only even prime.
- Even n > 2: divisible by 2, not prime.
- Check only odd divisors from 3 up to √n. If n = a × b with a ≤ b, then a ≤ √n, so we never need to check beyond √n.

Time complexity: O(√n).""",

    "C2": """from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fib(n - 1) + fib(n - 2)

# Manual dict alternative:
_memo = {}
def fib_manual(n):
    if n in _memo:
        return _memo[n]
    if n <= 1:
        return n
    _memo[n] = fib_manual(n - 1) + fib_manual(n - 2)
    return _memo[n]

Complexity comparison:
- Naive recursion: O(2^n) — each call branches into two, creating a tree of redundant subproblems.
- With memoisation: O(n) time, O(n) space — each value fib(0)…fib(n) is computed once and cached; subsequent calls return in O(1).""",

    "C3": """def two_sum(nums, target):
    seen = {}           # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

How O(n) is achieved:
For each element we compute its complement (target - num) and check the hash map in O(1) average time.
We make exactly one pass through the list, giving O(n) overall.
Space complexity is also O(n) for the hash map.""",

    "C4": """def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

Why O(log n):
Each iteration halves the remaining search space.
Starting with n elements, after k steps n/2^k elements remain.
We stop when n/2^k ≤ 1, i.e. k = log₂(n).
So at most ⌈log₂(n)⌉ comparisons are needed.""",

    "C5": """def is_balanced(s):
    stack = []
    matching = {')': '(', '}': '{', ']': '['}
    for ch in s:
        if ch in '({[':
            stack.append(ch)
        elif ch in ')}]':
            if not stack or stack[-1] != matching[ch]:
                return False
            stack.pop()
    return len(stack) == 0

Approach:
- Push every opening bracket onto the stack.
- For every closing bracket, check that the stack is non-empty and its top is the matching opener.
  If not, the string is unbalanced.
- At the end the stack must be empty; any remaining openers were never closed.

Time complexity: O(n) — one pass through the string with O(1) stack operations per character.
Space complexity: O(n) worst case (all openers).""",

    "C6": """class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def reverse_linked_list(head):
    prev = None
    curr = head
    while curr:
        next_node = curr.next   # save the next node before overwriting
        curr.next = prev        # reverse the link
        prev = curr             # advance prev
        curr = next_node        # advance curr
    return prev                 # prev is the new head

Walk-through with 1 → 2 → 3 → 4:
- Start: prev=None, curr=1
- Step 1: save 2, 1→None, prev=1, curr=2
- Step 2: save 3, 2→1,    prev=2, curr=3
- Step 3: save 4, 3→2,    prev=3, curr=4
- Step 4: save None, 4→3, prev=4, curr=None  → loop ends

Return prev=4. Result: 4 → 3 → 2 → 1. ✓""",

    "C7": """def permutations(lst):
    if len(lst) == 0:
        return [[]]
    result = []
    for i, elem in enumerate(lst):
        rest = lst[:i] + lst[i+1:]           # every element except the current one
        for perm in permutations(rest):       # all permutations of the remainder
            result.append([elem] + perm)
    return result

# permutations([1, 2, 3]) →
# [[1,2,3], [1,3,2], [2,1,3], [2,3,1], [3,1,2], [3,2,1]]

Recursive approach:
- Base case: the empty list has exactly one permutation — itself (the empty list).
- Recursive case: fix each element as the first element in turn, then recursively generate all
  permutations of the remaining elements and prepend the fixed element.

Time complexity: O(n × n!) — there are n! permutations each of length n, and building each takes O(n).""",

    "C8": """def merge_sorted(a, b):
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    result.extend(a[i:])    # append leftover elements from whichever list is not exhausted
    result.extend(b[j:])
    return result

Two-pointer approach:
- Maintain one pointer into each sorted list.
- At each step, append the smaller of the two current elements and advance that pointer.
- Once one list is exhausted, append all remaining elements from the other (already sorted).

Time complexity: O(m + n) — every element is visited exactly once.
Space complexity: O(m + n) for the output list.""",

    "C9": """from collections import deque

def shortest_path(grid, start, end):
    if not grid or grid[start[0]][start[1]] == 1 or grid[end[0]][end[1]] == 1:
        return -1
    rows, cols = len(grid), len(grid[0])
    queue = deque([(start, 0)])     # (position, steps taken)
    visited = {start}
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        (r, c), steps = queue.popleft()
        if (r, c) == end:
            return steps
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), steps + 1))
    return -1   # no path found

Why BFS guarantees the shortest path:
BFS explores nodes level by level — all cells reachable in k steps are processed before any cell reachable
in k+1 steps. Therefore the first time BFS reaches the destination it has taken the minimum possible steps.""",

    "C10": """import re

def calculate(expression):
    # Tokenise: extract integers and operators
    tokens = re.findall(r'\d+|[+\-*\/]{1,2}', expression.replace(' ', ''))

    # Single-pass stack approach respecting precedence (* and // before + and -)
    stack = []
    i = 0
    stack.append(int(tokens[0]))
    i = 1
    while i < len(tokens):
        op = tokens[i]
        next_num = int(tokens[i + 1])
        if op == '*':
            stack[-1] *= next_num
        elif op == '//':
            # Truncate toward zero (Python's int() on true division)
            stack[-1] = int(stack[-1] / next_num)
        elif op == '+':
            stack.append(next_num)
        elif op == '-':
            stack.append(-next_num)
        i += 2
    return sum(stack)

# Example: calculate('3 + 5 * 2')
# tokens = ['3', '+', '5', '*', '2']
# stack after '3':      [3]
# '+' 5  → push  5:     [3, 5]
# '*' 2  → 5*2=10:      [3, 10]
# sum([3, 10]) = 13  ✓

Approach:
Push numbers onto a stack. For * and //, perform the operation immediately on the stack top.
For + push the number; for - push its negation. Sum the stack at the end to collect all
additions/subtractions. This correctly handles operator precedence in a single O(n) pass.""",
}

assert len(gold_answers) == 30, f"Expected 30 gold answers, got {len(gold_answers)}"
print(f"Gold answers loaded: {len(gold_answers)} total "
      f"(G×5, M×5, D×10, C×10)")

## Step 3 — Run Base Models Locally

Qwen3-0.6B and Qwen3-1.7B are **not available on the HF Inference API free tier**.
Both models run locally via `transformers` on GPU (float16) or CPU (float32).
Each model is loaded once, all **30 prompts** run, then the model is unloaded before the next one.

In [12]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device} | dtype: {dtype}")

def _apply_chat_template(tokenizer, messages):
    """apply_chat_template returns a plain tensor in older transformers and
    a BatchEncoding dict in newer ones (4.50+). This handles both."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded

def run_model_local(model_id, prompts, hf_token, max_new_tokens=512):
    """Load model locally, run all prompts, unload. Uses GPU if available, else CPU."""
    print(f"  Loading {model_id} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else "cpu",
    )
    model.eval()

    responses = {}
    for p in prompts:
        print(f"  Prompt {p['id']}...", end=" ", flush=True)
        messages = [
            {"role": "system", "content": "You are a helpful reasoning assistant. Think step by step."},
            {"role": "user",   "content": p["prompt"]}
        ]
        input_ids = _apply_chat_template(tokenizer, messages).to(device)
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        responses[p["id"]] = response
        print("done")

    del model, tokenizer
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return responses


# Run both base models on all 10 prompts
base_responses = {}

for model_id in BASE_MODELS:
    print(f"\n── Running {model_id} ──")
    base_responses[model_id] = run_model_local(model_id, PROMPTS, HF_TOKEN)

print("\nAll base model responses collected.")

Using device: cuda | dtype: torch.float16

── Running Qwen/Qwen3-0.6B ──
  Loading Qwen/Qwen3-0.6B on cuda...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 872.55it/s]


  Prompt G1... done
  Prompt G2... done
  Prompt G3... done
  Prompt G4... done
  Prompt G5... done
  Prompt M1... done
  Prompt M2... done
  Prompt M3... done
  Prompt M4... done
  Prompt M5... done

── Running Qwen/Qwen3-1.7B ──
  Loading Qwen/Qwen3-1.7B on cuda...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 562.08it/s]


  Prompt G1... done
  Prompt G2... done
  Prompt G3... done
  Prompt G4... done
  Prompt G5... done
  Prompt M1... done
  Prompt M2... done
  Prompt M3... done
  Prompt M4... done
  Prompt M5... done

All base model responses collected.


## Step 4 — LLM-as-Judge Scoring via HF OpenAI-Compatible Router

Judge: `moonshotai/Kimi-K2-Instruct:novita` (different from ChatGPT gold generator ✓)

Scoring: 1–5 scale on reasoning quality
- **5**: Correct answer, clear step-by-step reasoning, no errors
- **4**: Correct answer, reasoning mostly clear with minor gaps
- **3**: Partially correct or reasoning has gaps
- **2**: Wrong answer but shows some reasoning attempt
- **1**: Wrong answer, no meaningful reasoning

In [13]:
JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY using this scale:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    """Score a response using Kimi through the HF OpenAI-compatible router."""
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question, gold=gold, response=response
    )
    last_error = None
    for attempt in range(3):
        try:
            completion = JUDGE_CLIENT.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=180,
                temperature=0.0,
            )
            out = completion.choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as error:
            last_error = error
            time.sleep(2 * (attempt + 1))
    return 0, f"ERROR after retries: {last_error}"


print(f"Judge helper version: hf-router-openai | model={JUDGE_MODEL}")


# If the kernel was restarted, reuse already-generated responses from the previous CSV.
if "base_responses" not in globals() and os.path.exists("baseline_results.csv"):
    old_df = pd.read_csv("baseline_results.csv")
    if {"model", "prompt_id", "response"}.issubset(old_df.columns):
        base_responses = {
            model_id: dict(zip(group["prompt_id"], group["response"]))
            for model_id, group in old_df.groupby("model")
        }
        print("Loaded existing model responses from baseline_results.csv; rescoring only.")
if "base_responses" not in globals():
    raise RuntimeError("No saved/generated base_responses found. Run Step 3 once to generate Qwen responses, then rerun Step 4/5.")

# Score all responses
records = []
for model_id in BASE_MODELS:
    print(f"\n── Judging {model_id} ──")
    for p in PROMPTS:
        print(f"  Prompt {p['id']}...", end=" ")
        response = base_responses[model_id][p["id"]]
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], response)
        records.append({
            "model":    model_id,
            "stage":    "baseline",
            "trial":    "base",
            "prompt_id": p["id"],
            "type":     p["type"],
            "response": response,
            "score":    score,
            "reason":   reason,
        })
        print(f"score={score}")
        time.sleep(1)

print("\nJudging complete.")

Judge helper version: chat-first v3 | model=moonshotai/Kimi-K2-Instruct

── Judging Qwen/Qwen3-0.6B ──
  Prompt G1... score=3
  Prompt G2... score=5
  Prompt G3... score=3
  Prompt G4... score=4
  Prompt G5... score=2
  Prompt M1... score=5
  Prompt M2... score=5
  Prompt M3... score=5
  Prompt M4... score=5
  Prompt M5... score=5

── Judging Qwen/Qwen3-1.7B ──
  Prompt G1... score=4
  Prompt G2... score=5
  Prompt G3... score=4
  Prompt G4... score=5
  Prompt G5... score=5
  Prompt M1... score=5
  Prompt M2... score=5
  Prompt M3... score=5
  Prompt M4... score=5
  Prompt M5... score=5

Judging complete.


## Step 5 — Results Summary

In [ ]:
df = pd.DataFrame(records)

# Overall summary
summary = df.groupby("model")["score"].agg(["mean", "sum", "count"]).round(2)
summary.columns = ["avg_score", "total_score", "n_prompts"]
print("\n── Baseline Summary (all 30 prompts) ──")
print(summary.to_string())

# Per-type breakdown — shows which category each model struggles on
print("\n── Avg score by prompt type ──")
type_summary = df.groupby(["model", "type"])["score"].mean().round(2).unstack()
print(type_summary.to_string())

# Full per-prompt pivot
print("\n── Per-prompt scores ──")
pivot = df.pivot_table(index="prompt_id", columns="model", values="score")
print(pivot.to_string())

# Save
df.to_csv("baseline_results.csv", index=False)
with open("gold_answers.json", "w") as f:
    json.dump(gold_answers, f, indent=2)
with open("prompts.json", "w") as f:
    json.dump(PROMPTS, f, indent=2)

print("\nSaved: baseline_results.csv, gold_answers.json, prompts.json")
print("These files are needed by notebooks 01, 02, and 03.")
print("\nType breakdown guides dataset choice:")
print("  High coding score → Dataset A (reasoning traces) likely better")
print("  High math score   → Dataset B (math solutions) likely better")